# Data Quality Check
This notebook derives each player's current and peak valuation, checks how up-to-date those valuations are and filters out players whose data is too stale

In [4]:
import pandas as pd

player_valuations = pd.read_csv('../data/player_valuations.csv')
players = pd.read_csv('../data/players.csv')
player_valuations['date'] = pd.to_datetime(player_valuations['date'])

print(player_valuations.shape, player_valuations.columns.tolist())
print(players.shape, players.columns.tolist())

(507815, 6) ['player_id', 'date', 'market_value_in_eur', 'current_club_name', 'current_club_id', 'player_club_domestic_competition_id']
(50149, 26) ['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']


In [5]:
current = player_valuations.sort_values('date').groupby('player_id').tail(1)[['player_id', 'market_value_in_eur', 'date']].rename(columns={'market_value_in_eur': 'current_value', 'date': 'current_date'})

peak_idx = player_valuations.groupby('player_id')['market_value_in_eur'].idxmax()
peak = player_valuations.loc[peak_idx, ['player_id', 'market_value_in_eur', 'date']].rename(columns={'market_value_in_eur': 'peak_value', 'date': 'peak_date'})


player_values = current.merge(peak, on='player_id').merge(players[['player_id', 'name']], on='player_id')

max_date = player_valuations['date'].max()
player_values['days_since_update'] = (max_date - player_values['current_date']).dt.days
player_values['is_stale'] = player_values['days_since_update'] > 365

print(f"Dataset max date: {max_date.date()}")
print(player_values['days_since_update'].describe())
print(player_values['is_stale'].value_counts())

Dataset max date: 2026-02-27
count    31507.000000
mean      1273.394420
std       1136.365299
min          0.000000
25%        633.000000
50%        977.000000
75%       1644.500000
max       7392.000000
Name: days_since_update, dtype: float64
is_stale
True     25502
False     6005
Name: count, dtype: int64


In [6]:
player_values_fresh = player_values[~player_values['is_stale']].copy()

print(f"Players before filtering: {len(player_values)}")
print(f"Players after freshness filter: {len(player_values_fresh)}")

player_values_fresh.to_csv('../data/player_values_fresh.csv', index=False)
player_values_fresh.sort_values('current_value', ascending=False).head(10)

Players before filtering: 31507
Players after freshness filter: 6005


,player_id,current_value,current_date,peak_value,peak_date,name,days_since_update,is_stale
26914,418560,200000000,2025-12-09,200000000,2024-10-01,Erling Haaland,80,False
27696,581678,160000000,2025-12-12,180000000,2023-12-22,Jude Bellingham,77,False
30155,580195,130000000,2025-12-19,140000000,2024-12-20,Jamal Musiala,70,False
30164,566723,130000000,2025-12-19,130000000,2025-10-14,Michael Olise,70,False
26845,433177,130000000,2025-12-09,150000000,2024-12-16,Bukayo Saka,80,False
26569,568177,120000000,2025-12-09,130000000,2024-12-16,Cole Palmer,80,False
26964,349066,120000000,2025-12-09,140000000,2025-10-17,Alexander Isak,80,False
28033,369081,120000000,2025-12-12,130000000,2024-10-11,Federico Valverde,77,False
26896,357662,120000000,2025-12-09,120000000,2024-05-27,Declan Rice,80,False
28465,487469,110000000,2025-12-15,110000000,2025-12-15,Vitinha,74,False
